# **Network Visualization & Centrality Measures**

## **Install Packages**

In [ ]:
install.packages("igraph")

## **Raw Data**

In [ ]:
table <- read.csv("SOCIAL NETWORK ANALYSIS QUESTIONNAIRE.csv", header=T, as.is=T)

head(table)
nrow(table)
ncol(table)

### **Student Name with IDs**

In [ ]:
# Extract all student names from main table
all_students <- unique(c(table$Full.Name,
                         table$Friend.1, table$Friend.2, table$Friend.3,
                         table$Friend.4, table$Friend.5, table$Friend.6,
                         table$Friend.7, table$Friend.8, table$Friend.9,
                         table$Friend.10, table$Tutor.1, table$Tutor.2,
                         table$Tutor.3, table$Tutor.4, table$Tutor.5,
                         table$Tutor.6, table$Tutor.7, table$Tutor.8,
                         table$Tutor.9, table$Tutor.10))

# Remove "NO FRIENDS" from the list
all_students <- all_students[all_students != "NO FRIENDS"]

# Create the list
student <- data.frame(
  ID = table$ID,
  Name = all_students)

student

## **Friendship Link**

### Data Extraction

In [ ]:
#Extracting friendship link

friendship <- data.frame(
  From = rep(table$Full.Name, 10), # Repeat 'Full Name' 10 times for each of the 10 friend columns
  To = c(table$`Friend.1`, table$`Friend.2`, table$`Friend.3`, table$`Friend.4`, table$`Friend.5`,
         table$`Friend.6`, table$`Friend.7`, table$`Friend.8`, table$`Friend.9`, table$`Friend.10`))

friendship

In [ ]:
# Remove rows where 'To' is "NO FRIENDS"
friendship <- friendship[friendship$To != "NO FRIENDS", ]

friendship

In [ ]:
#Reorder in alphabetical

friendship <- friendship[order(friendship$From), ]
row.names(friendship) <- NULL  # Reset row names

friendship

In [ ]:
# Match the names in 'friendship' to the IDs in 'student'
friendship_numbered <- data.frame(
  From_ID = match(friendship$From, student$Name),
  To_ID   = match(friendship$To, student$Name))

# Display the result (From–To table using ID)
friendship_numbered

In [ ]:
library(igraph)

g <- graph_from_data_frame(d=friendship_numbered, directed=T)
friendship_numbered <- as.matrix(as_adjacency_matrix(g))

friendship_numbered

In [ ]:
write.csv(friendship_numbered, file="friendship_edges.csv")

### **Gender Assignment**

In [ ]:
ordered_ids <- as.numeric(V(g)$name)

# Get the student names in the correct order
namesf <- student$Name[ordered_ids]

# Get the gender for each of those names from the original 'table'
genderf <- table$Gender[match(student$Name[ordered_ids], table$Full.Name)]

# Now, create the correctly ordered shape and color vectors
gshapef <- ifelse(genderf == "Male (M)", "circle", "square")
gcolorsf <- ifelse(genderf == "Male (M)", "#A2D2FF", "pink")


In [ ]:
gender_table <- data.frame(ID=student$ID, Name=namesf, Gender=genderf, Shape=gshapef, Colors=gcolorsf)
gender_table

### **Middle Name**

In [ ]:
# 1. Get the full names in the correct order
ordered_full_names <- student$Name[as.numeric(V(g)$name)]

# 2. Extract the second word (middle name) from each name
middle_names <- sapply(ordered_full_names, function(name) {
  words <- strsplit(trimws(name), " ")[[1]]
  return(words[2])
})

# 3. Find which middle names are duplicated
duplicated_middle_names <- names(which(table(middle_names) > 1))

# 4. Create the final labels vector
final_labels <- sapply(ordered_full_names, function(name) {
  words <- strsplit(trimws(name), " ")[[1]]
  middle_name <- words[2]

  # If the middle name is in our list of duplicates...
  if (middle_name %in% duplicated_middle_names) {
    # ...create a "FirstName M." label
    first_name <- words[1]
    middle_initial <- substr(middle_name, 1, 1)
    return(paste0(first_name, " ", middle_initial, "."))
  } else {
    # ...otherwise, just use the middle name
    return(middle_name)
  }
})

mname_table <- data.frame(Middle_Name=final_labels)
mname_table

### Network Analysis

In [ ]:
#Indegree centrality = most popular

degree_g <- degree(g, mode="all")
indegree_g <- degree(g, mode="in")
outdegree_g <- degree(g,mode="out")

degree_table <- data.frame(Name= namesf, Student_ID = student$ID, General=degree_g, Indegree=indegree_g, Outdegree=outdegree_g)
degree_table

In [ ]:
#Betweenness centrality = bridges

betweenness_g <- betweenness(g, directed=F, weights=NA, normalized = TRUE)

In [ ]:
#Closeness centrality = distance to others (how fast information spreads)

closeness_g <- closeness(g, mode="all", weights=NA)

In [ ]:
#Full network analysis

centrality_table <- data.frame(Name= namesf, Student_ID = student$ID,
                               Degree=degree_g, Indegree=indegree_g,
                               Outdegree=outdegree_g, Betweenness=betweenness_g,
                               Closeness=closeness_g)
centrality_table

### Network Visualization



#### **1. Name**

##### **a. Degree**

In [ ]:
plot(g,
     main="Friendship Network (Degree)",
     layout=layout_nicely,
     edge.curve=0.5,
     vertex.size=degree_g * 0.5,
     edge.width=0.6,
     edge.color="grey",
     vertex.label=final_labels,
     vertex.label.color="black",
     vertex.label.font=2,
     vertex.label.cex=0.4,
     edge.arrow.size=0,
     vertex.shape=gshapef,
     vertex.color=gcolorsf)

     legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("pink", "#A2D2FF"),
       pt.cex=2,
       bty="n",
       title="Gender")


In [ ]:
g_size <- ifelse(degree_g > 20, degree_g*0.7, degree_g*0.4)

plot(g,
     main="Friendship Network (Degree)",
     layout = layout.circle,
     edge.curve = 0.1,
     vertex.size = g_size,
     edge.width = 0.6,
     edge.color = "grey",
     vertex.label = final_labels,
     vertex.label.color = "black",
     vertex.label.font = 2,
     vertex.label.cex = 0.4,
     edge.arrow.size = 0,
     vertex.shape = gshapef,
     vertex.color = gcolorsf)

legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("pink", "#A2D2FF"),
       pt.cex = 2,
       bty="n",
       title="Gender")

##### **b. Closeness**

In [ ]:
plot(g,
     main="Friendship Network (Closeness)",
     layout=layout_nicely,
     edge.curve=0.5,
     vertex.size=closeness_g * 1000,
     edge.width=0.6,
     edge.color="grey",
     vertex.label=final_labels,
     vertex.label.font=2,
     vertex.label.color="black",
     vertex.label.cex=0.4,
     edge.arrow.size=0.3,
     vertex.shape=gshapef,
     vertex.color=gcolorsf)

     legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("pink", "#A2D2FF"),
       pt.cex=2,
       bty="n",
       title="Gender")

In [ ]:
gc_size <- ifelse(closeness_g > 0.0116, closeness_g*1000, closeness_g*500)

plot(g,
     main="Friendship Network (Closeness)",
     layout = layout.circle,
     edge.curve = 0.1,
     vertex.size=gc_size,
     edge.width=0.6,
     edge.color="grey",
     vertex.label=final_labels,
     vertex.label.font=2,
     vertex.label.color="black",
     vertex.label.cex=0.4,
     edge.arrow.size=0.3,
     vertex.shape=gshapef,
     vertex.color=gcolorsf)

     legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("pink", "#A2D2FF"),
       pt.cex=2,
       bty="n",
       title="Gender")

##### **c. Betweenness**

In [ ]:
plot(g,
     main="Friendship Network (Betweenness)",
     layout=layout_nicely,
     edge.curve=0.5,
     vertex.size=betweenness_g * 100,
     edge.width=0.6,
     edge.color="grey",
     vertex.label=final_labels,
     vertex.label.font=2,
     vertex.label.color="black",
     vertex.label.cex=0.4,
     edge.arrow.size=0.3,
     vertex.shape=gshapef,
     vertex.color=gcolorsf)

     legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("pink", "#A2D2FF"),
       pt.cex=2,
       bty="n",
       title="Gender")

In [ ]:
gb_size <- ifelse(betweenness_g > 0.0182, betweenness_g*100, betweenness_g*50)

plot(g,
     main="Friendship Network (Betweenness)",
     layout=layout.circle,
     edge.curve=0.1,
     vertex.size=gb_size,
     edge.width=0.6,
     edge.color="grey",
     vertex.label=final_labels,
     vertex.label.font=2,
     vertex.label.color="black",
     vertex.label.cex=0.4,
     edge.arrow.size=0.3,
     vertex.shape=gshapef,
     vertex.color=gcolorsf)

     legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("pink", "#A2D2FF"),
       pt.cex=2,
       bty="n",
       title="Gender")

#### **2. Degree**

In [ ]:
plot(g,
     main="Friendship Network (Degree)",
     layout=layout_nicely,
     edge.curve=0.5,
     vertex.size=degree_g * 0.5,
     edge.width=0.6,
     edge.color="grey",
     vertex.label.color="black",
     vertex.label.font=2,
     edge.arrow.size=0,
     vertex.shape=gshapef,
     vertex.color=gcolorsf)

     legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("pink", "#A2D2FF"),
       pt.cex=2,
       bty="n",
       title="Gender")

In [ ]:
plot(g,
     main="Friendship Network (Degree)",
     layout=layout_nicely,
     edge.curve=0.5,
     vertex.size=degree_g * 0.5,
     edge.width=0.6,
     edge.color="grey",
     vertex.label.color="black",
     edge.arrow.size=0,
     vertex.shape=gshapef,
     vertex.color=gcolorsf)

     legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("pink", "#A2D2FF"),
       pt.cex=2,
       bty="n",
       title="Gender")

#### **3. Indegree**

In [ ]:
plot(g,
     main="Friendship Network (Indegree)",
     layout=layout_nicely,
     edge.curve=0.5,
     vertex.size=indegree_g * 0.8,
     edge.width=0.6,
     edge.color="grey",
     vertex.label=final_labels,
     vertex.label.cex=0.5,
     vertex.label.color="black",
     edge.arrow.size=0.3,
     edge.arrow.mode="->",
     vertex.shape=gshapef,
     vertex.color=gcolorsf)

     legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("pink", "#A2D2FF"),
       pt.cex=2,
       bty="n",
       title="Gender")

In [ ]:
gi_size <- ifelse(indegree_g > 12, indegree_g*1, indegree_g*0.5)

plot(g,
     main="Friendship Network (Indegree)",
     layout = layout.circle,
     edge.curve = 0.1,
     vertex.size = gi_size,
     edge.width = 0.6,
     edge.color = "grey",
     vertex.label.color = "black",
     vertex.label.font = 2,
     vertex.label.cex = 1,
     edge.arrow.size = 0,
     vertex.shape = gshapef,
     vertex.color = gcolorsf)

legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("pink", "#A2D2FF"),
       pt.cex = 2,
       bty="n",
       title="Gender")

In [ ]:
gi_size <- ifelse(indegree_g > 12, indegree_g*1, indegree_g*0.5)

plot(g,
     main="Friendship Network (Indegree)",
     layout = layout.circle,
     edge.curve = 0.1,
     vertex.size = gi_size,
     edge.width = 0.6,
     edge.color = "grey",
     vertex.label=final_labels,
     vertex.label.color = "black",
     vertex.label.font = 2,
     vertex.label.cex = 0.4,
     edge.arrow.size = 0,
     vertex.shape = gshapef,
     vertex.color = gcolorsf)

legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("pink", "#A2D2FF"),
       pt.cex = 2,
       bty="n",
       title="Gender")

#### **4. Outdegree**

In [ ]:
plot(g,
     main="Friendship Network (Outdegree)",
     layout=layout_nicely,
     edge.curve=0.5,
     vertex.size=outdegree_g * 1,
     edge.width=0.6,
     edge.color="grey",
     vertex.label.color="black",
     edge.arrow.size=0.3,
     edge.arrow.mode="<-",
     vertex.shape=gshapef,
     vertex.color=gcolorsf)

     legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("pink", "#A2D2FF"),
       pt.cex=2,
       bty="n",
       title="Gender")

In [ ]:
go_size <- ifelse(outdegree_g >= 10, outdegree_g*1.2, outdegree_g*0.6)

plot(g,
     main="Friendship Network (Outdegree)",
     layout = layout.circle,
     edge.curve = 0.1,
     vertex.size = go_size,
     edge.width = 0.6,
     edge.color = "grey",
     vertex.label = final_labels,
     vertex.label.color = "black",
     vertex.label.font = 2,
     vertex.label.cex = 0.4,
     edge.arrow.size = 0,
     vertex.shape = gshapef,
     vertex.color = gcolorsf)

legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("pink", "#A2D2FF"),
       pt.cex = 2,
       bty="n",
       title="Gender")

In [ ]:
go_size <- ifelse(outdegree_g >= 10, outdegree_g*1.2, outdegree_g*0.6)

plot(g,
     main="Friendship Network (Outdegree)",
     layout = layout.circle,
     edge.curve = 0.1,
     vertex.size = go_size,
     edge.width = 0.6,
     edge.color = "grey",
     vertex.label.color = "black",
     vertex.label.font = 2,
     vertex.label.cex = 1,
     edge.arrow.size = 0,
     vertex.shape = gshapef,
     vertex.color = gcolorsf)

legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("pink", "#A2D2FF"),
       pt.cex = 2,
       bty="n",
       title="Gender")

#### **5. Closeness**

In [ ]:
plot(g,
     main="Friendship Network (Closeness)",
     layout=layout_nicely,
     edge.curve=0.5,
     vertex.size=closeness_g * 1000,
     edge.width=0.6,
     edge.color="grey",
     vertex.label.color="black",
     edge.arrow.size=0.3,
     vertex.shape=gshapef,
     vertex.color=gcolorsf)

     legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("pink", "#A2D2FF"),
       pt.cex=2,
       bty="n",
       title="Gender")

#### **6. Betweenness**

In [ ]:
plot(g,
     main="Friendship Network (Betweenness)",
     layout=layout_nicely,
     edge.curve=0.5,
     vertex.size=betweenness_g * 100,
     edge.width=0.6,
     edge.color="grey",
     vertex.label.color="black",
     edge.arrow.size=0.1,
     vertex.shape=gshapef,
     vertex.color=gcolorsf)

     legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("pink", "#A2D2FF"),
       pt.cex=2,
       bty="n",
       title="Gender")

## **Peer-tutoring Link**

### Data Extraction

In [ ]:
#Extracting peer-tutoring links

tutor <- data.frame(
  From = rep(table$Full.Name, 10), # Repeat 'Full Name' 10 times for each of the 10 tutor columns
  To = c(table$Tutor.1, table$Tutor.2, table$Tutor.3, table$Tutor.4, table$Tutor.5,
         table$Tutor.6, table$Tutor.7, table$Tutor.8, table$Tutor.9, table$Tutor.10))

# Remove rows where 'To' is "NO FRIENDS"
tutor <- tutor[tutor$To != "NO FRIENDS", ]

tutor

In [ ]:
#Reorder in alphabetical

tutor <- tutor[order(tutor$From), ]
row.names(tutor) <- NULL  # Reset row names

tutor

In [ ]:
# Match the names in 'tutor' to the IDs in 'student'
tutor_numbered <- data.frame(
  From_ID = match(tutor$From, student$Name),
  To_ID   = match(tutor$To, student$Name))

# Display the result (From–To table using ID)
tutor_numbered

In [ ]:
library(igraph)

v <- graph_from_data_frame(d=tutor_numbered, directed=T)
tutor_numbered <- as.matrix(as_adjacency_matrix(v))

tutor_numbered

In [ ]:
write.csv(tutor_numbered, file="tutor_edges.csv")

### **Gender Assignment**

In [ ]:
ordered_ids <- as.numeric(V(v)$name)

# Get the student names in the correct order
namest <- student$Name[ordered_ids]

# Get the gender for each of those names from the original 'table'
gendert <- table$Gender[match(student$Name[ordered_ids], table$Full.Name)]

# Now, create the correctly ordered shape and color vectors
gshapet <- ifelse(gendert == "Male (M)", "circle", "square")
gcolorst <- ifelse(gendert == "Male (M)", "#FAFAD2", "#E0BBE4")


In [ ]:
gender_table <- data.frame(ID=student$ID, Name=namest, Gender=gendert, Shape=gshapet, Colors=gcolorst)
gender_table

### **Middle Name**

In [ ]:
# 1. Get the full names in the correct order
ordered_full_namest <- student$Name[as.numeric(V(v)$name)]

# 2. Extract the second word (middle name) from each name
middle_namest <- sapply(ordered_full_namest, function(name) {
  words <- strsplit(trimws(name), " ")[[1]]
  return(words[2])
})

# 3. Find which middle names are duplicated
duplicated_middle_namest <- names(which(table(middle_namest) > 1))

# 4. Create the final labels vector
final_labelst <- sapply(ordered_full_namest, function(name) {
  words <- strsplit(trimws(name), " ")[[1]]
  middle_name <- words[2]

  # If the middle name is in our list of duplicates...
  if (middle_name %in% duplicated_middle_namest) {
    # ...create a "FirstName M." label
    first_name <- words[1]
    middle_initial <- substr(middle_name, 1, 1)
    return(paste0(first_name, " ", middle_initial, "."))
  } else {
    # ...otherwise, just use the middle name
    return(middle_name)
  }
})

mname_tablet <- data.frame(Middle_Name=final_labelst)
mname_tablet

### Network Analysis

In [ ]:
#Indegree centrality = most popular

degree_v <- degree(v, mode="all")
indegree_v <- degree(v, mode="in")
outdegree_v <- degree(v,mode="out")

degree_table <- data.frame(Name= namest, Student_ID = student$ID, General=degree_v, Indegree=indegree_v, Outdegree=outdegree_v)
degree_table

In [ ]:
#Betweenness centrality = bridges

betweenness_v <- betweenness(v, directed=F, weights=NA, normalized = TRUE)

In [ ]:
#Closeness centrality = distance to others (how fast information spreads)

closeness_v <- closeness(v, mode="all", weights=NA)

In [ ]:
#Full network analysis

centrality_table <- data.frame(Name= namest, Student_ID = student$ID,
                               Degree=degree_v, Indegree=indegree_v,
                               Outdegree=outdegree_v, Betweenness=betweenness_v,
                               Closeness=closeness_v)
centrality_table

### Network Visualization

#### **1. Name**

##### **a. Degree**

In [ ]:
plot(v,
     main="Peer-Tutor Network (Degree)",
     layout=layout_with_lgl,
     vertex.size=degree_v * 0.7,
     edge.width=0.6,
     edge.color="grey",
     vertex.label=final_labelst,
     vertex.label.color="black",
     vertex.label.font=2,
     vertex.label.cex=0.5,
     edge.arrow.size=0,
     vertex.shape=gshapet,
     vertex.color=gcolorst)

     legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("#E0BBE4", "#FAFAD2"),
       pt.cex=2,
       bty="n",
       title="Gender")

In [ ]:
v_size <- ifelse(degree_v > 15, degree_v*0.7, degree_v*0.4)

plot(g,
     main="Peer-Tutor Network (Degree)",
     layout = layout.circle,
     edge.curve = 0.1,
     vertex.size = v_size,
     edge.width = 0.6,
     edge.color = "grey",
     vertex.label = final_labels,
     vertex.label.color = "black",
     vertex.label.font = 2,
     vertex.label.cex = 0.4,
     edge.arrow.size = 0,
     vertex.shape = gshapet,
     vertex.color = gcolorst)

legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("#E0BBE4", "#FAFAD2"),
       pt.cex=2,
       bty="n",
       title="Gender")

##### **b. Closeness**

In [ ]:
plot(v,
     main="Peer-Tutor Network (Closeness)",
     layout=layout_nicely,
     edge.curve=0.5,
     vertex.size=closeness_v * 1000,
     edge.width=0.6,
     edge.color="grey",
     vertex.label=final_labelst,
     vertex.label.color="black",
     vertex.label.font=2,
     vertex.label.cex=0.5,
     edge.arrow.size=0.3,
     vertex.shape=gshapet,
     vertex.color=gcolorst)

     legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("#E0BBE4", "#FAFAD2"),
       pt.cex=2,
       bty="n",
       title="Gender")

In [ ]:
vc_size <- ifelse(closeness_v > 0.0108, closeness_v*1000, closeness_v*500)

plot(v,
     main="Peer-Tutor Network (Closeness)",
     layout=layout.circle,
     edge.curve=0.1,
     vertex.size=vc_size,
     edge.width=0.6,
     edge.color="grey",
     vertex.label=final_labelst,
     vertex.label.color="black",
     vertex.label.font=2,
     vertex.label.cex=0.5,
     edge.arrow.size=0.3,
     vertex.shape=gshapet,
     vertex.color=gcolorst)

     legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("#E0BBE4", "#FAFAD2"),
       pt.cex=2,
       bty="n",
       title="Gender")

##### **c. Betweenness**

In [ ]:
plot(v,
     main="Peer-Tutor Network (Betweenness)",
     layout=layout_nicely,
     edge.curve=0.5,
     vertex.size=betweenness_v * 90,
     edge.width=0.6,
     edge.color="grey",
     vertex.label=final_labelst,
     vertex.label.color="black",
     vertex.label.font=2,
     vertex.label.cex=0.5,
     edge.arrow.size=0.3,
     vertex.shape=gshapet,
     vertex.color=gcolorst)

     legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("#E0BBE4", "#FAFAD2"),
       pt.cex=2,
       bty="n",
       title="Gender")

In [ ]:
vb_size <- ifelse(betweenness_v >0.0606, betweenness_v*90, betweenness_v*45)

plot(v,
     main="Peer-Tutor Network (Betweenness)",
     layout=layout.circle,
     edge.curve=0.1,
     vertex.size=vb_size,
     edge.width=0.6,
     edge.color="grey",
     vertex.label=final_labelst,
     vertex.label.color="black",
     vertex.label.font=2,
     vertex.label.cex=0.5,
     edge.arrow.size=0.3,
     vertex.shape=gshapet,
     vertex.color=gcolorst)

     legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("#E0BBE4", "#FAFAD2"),
       pt.cex=2,
       bty="n",
       title="Gender")

#### **2. Degree**

In [ ]:
plot(v,
     main="Peer-Tutor Network",
     layout=layout_with_kk,
     vertex.size=degree_v * 0.6,
     edge.width=0.6,
     edge.color="grey",
     vertex.label.color="black",
     vertex.label.font=2,
     vertex.label.cex=1,
     edge.arrow.size=0,
     vertex.shape=gshapet,
     vertex.color=gcolorst)

     legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("#E0BBE4", "#FAFAD2"),
       pt.cex=2,
       bty="n",
       title="Gender")

In [ ]:
plot(v,
     main="Peer-Tutor Network (Degree)",
     layout=layout_with_kk,
     vertex.size=degree_v * 0.6,
     edge.width=0.6,
     edge.color="grey",
     vertex.label.color="black",
     vertex.label.cex=1,
     edge.arrow.size=0,
     vertex.shape=gshapet,
     vertex.color=gcolorst)

     legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("#E0BBE4", "#FAFAD2"),
       pt.cex=2,
       bty="n",
       title="Gender")

#### **3. Indegree**

In [ ]:
plot(v,
     main="Peer-Tutor Network (Indegree)",
     layout=layout_nicely,
     edge.curve=0.5,
     vertex.size=indegree_v * 1,
     edge.width=0.6,
     edge.color="grey",
     vertex.label.color="black",
     vertex.label.cex=0.85,
     edge.arrow.size=0.3,
     arrow.mode=2,
     vertex.shape=gshapet,
     vertex.color=gcolorst)

     legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("#E0BBE4", "#FAFAD2"),
       pt.cex=2,
       bty="n",
       title="Gender")

In [ ]:
vi_size <- ifelse(indegree_v > 8, indegree_v*1.1, indegree_v*0.7)

plot(g,
     main="Peer-Tutor Network (Indegree)",
     layout = layout.circle,
     edge.curve = 0.1,
     vertex.size = vi_size,
     edge.width = 0.6,
     edge.color = "grey",
     vertex.label = final_labels,
     vertex.label.color = "black",
     vertex.label.font = 2,
     vertex.label.cex = 0.4,
     edge.arrow.size = 0,
     vertex.shape = gshapet,
     vertex.color = gcolorst)

legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("#E0BBE4", "#FAFAD2"),
       pt.cex=2,
       bty="n",
       title="Gender")

In [ ]:
v_size <- ifelse(degree_v > 15, degree_v*0.7, degree_v*0.4)

plot(g,
     main="Peer-Tutor Network (Indegree)",
     layout = layout.circle,
     edge.curve = 0.1,
     vertex.size = v_size,
     edge.width = 0.6,
     edge.color = "grey",
     vertex.label.color = "black",
     vertex.label.font = 2,
     vertex.label.cex = 1,
     edge.arrow.size = 0,
     vertex.shape = gshapet,
     vertex.color = gcolorst)

legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("#E0BBE4", "#FAFAD2"),
       pt.cex=2,
       bty="n",
       title="Gender")

#### **4. Outdegree**

In [ ]:
plot(v,
     main="Peer-Tutor Network (Outdegree)",
     layout=layout_nicely,
     edge.curve=0.5,
     vertex.size=outdegree_v * 1.2,
     edge.width=0.6,
     edge.color="grey",
     vertex.label.color="black",
     vertex.label.cex=1,
     edge.arrow.size=0.3,
     edge.arrow.mode=1,
     vertex.shape=gshapet,
     vertex.color=gcolorst)

     legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("#E0BBE4", "#FAFAD2"),
       pt.cex=2,
       bty="n",
       title="Gender")

In [ ]:
vo_size <- ifelse(outdegree_v > 9, outdegree_v*1.1, outdegree_v*0.7)

plot(g,
     main="Peer-Tutor Network (Outdegree)",
     layout = layout.circle,
     edge.curve = 0.1,
     vertex.size = vo_size,
     edge.width = 0.6,
     edge.color = "grey",
     vertex.label = final_labels,
     vertex.label.color = "black",
     vertex.label.font = 2,
     vertex.label.cex = 0.4,
     edge.arrow.size = 0,
     vertex.shape = gshapet,
     vertex.color = gcolorst)

legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("#E0BBE4", "#FAFAD2"),
       pt.cex=2,
       bty="n",
       title="Gender")

In [ ]:
vo_size <- ifelse(outdegree_v > 9, outdegree_v*1.1, outdegree_v*0.7)

plot(g,
     main="Peer-Tutor Network (Outdegree)",
     layout = layout.circle,
     edge.curve = 0.1,
     vertex.size = vo_size,
     edge.width = 0.6,
     edge.color = "grey",
     vertex.label.color = "black",
     vertex.label.font = 2,
     vertex.label.cex = 1,
     edge.arrow.size = 0,
     vertex.shape = gshapet,
     vertex.color = gcolorst)

legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("#E0BBE4", "#FAFAD2"),
       pt.cex=2,
       bty="n",
       title="Gender")

#### **5. Closeness**

In [ ]:
plot(v,
     main="Peer-Tutor Network (Closeness)",
     layout=layout_nicely,
     edge.curve=0.5,
     vertex.size=closeness_v * 1000,
     edge.width=0.6,
     edge.color="grey",
     vertex.label.color="black",
     vertex.label.cex=1,
     edge.arrow.size=0.3,
     vertex.shape=gshapet,
     vertex.color=gcolorst)

     legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("#E0BBE4", "#FAFAD2"),
       pt.cex=2,
       bty="n",
       title="Gender")

#### **6. Betweenness**

In [ ]:
plot(v,
     main="Peer-Tutor Network (Betweenness)",
     layout=layout_nicely,
     edge.curve=0.5,
     vertex.size=betweenness_v * 100,
     edge.width=0.6,
     edge.color="grey",
     vertex.label.color="black",
     vertex.label.cex=1,
     edge.arrow.size=0.3,
     edge.arrow.mode="->",
     vertex.shape=gshapet,
     vertex.color=gcolorst)

     legend("bottomleft",
       legend=c("Female", "Male"),
       pch=c(22, 21),
       pt.bg=c("#E0BBE4", "#BDE0FE"),
       pt.cex=2,
       bty="n",
       title="Gender")

# **The Top 5**

## **Friendship**

In [ ]:
# Create a data frame for friendship network centrality
friendship_centrality <- data.frame(
  Name = namesf,
  Student_ID = student$ID,
  Degree = degree_g,
  Betweenness = betweenness_g,
  Closeness = closeness_g
)

# Display top 5 for each centrality measure in the friendship network
print("Top 5 in Friendship Network:")

print("Top 5 by Degree:")
print(friendship_centrality[order(-friendship_centrality$Degree), ][1:5, c("Name", "Degree")])

print("Top 5 by Betweenness:")
print(friendship_centrality[order(-friendship_centrality$Betweenness), ][1:5, c("Name", "Betweenness")])

print("Top 5 by Closeness:")
print(friendship_centrality[order(-friendship_centrality$Closeness), ][1:5, c("Name", "Closeness")])

## **Peer-Tutor**

In [ ]:
# Create a data frame for peer-tutor network centrality
tutor_centrality <- data.frame(
  Name = namest,
  Student_ID = student$ID,
  Degree = degree_v,
  Betweenness = betweenness_v,
  Closeness = closeness_v
)

# Display top 5 for each centrality measure in the peer-tutor network
print("Top 5 in Peer-Tutor Network:")

print("Top 5 by Degree:")
print(tutor_centrality[order(-tutor_centrality$Degree), ][1:5, c("Name", "Degree")])

print("Top 5 by Betweenness:")
print(tutor_centrality[order(-tutor_centrality$Betweenness), ][1:5, c("Name", "Betweenness")])

print("Top 5 by Closeness:")
print(tutor_centrality[order(-tutor_centrality$Closeness), ][1:5, c("Name", "Closeness")])

## **Comparison**

### **Degree**

In [ ]:
degree_comparison <- data.frame(Friendship=friendship_centrality[order(-friendship_centrality$Degree), ][1:5, c("Name", "Degree")],
                                Peer_Tutor=tutor_centrality[order(-tutor_centrality$Degree), ][1:5, c("Name", "Degree")])

degree_comparison

### **Closeness**

In [ ]:
closeness_comparison <- data.frame(Friendship=friendship_centrality[order(-friendship_centrality$Closeness), ][1:5, c("Name", "Closeness")],
                                Peer_Tutor=tutor_centrality[order(-tutor_centrality$Closeness), ][1:5, c("Name", "Closeness")])

closeness_comparison

### **Betweenness**

In [ ]:
betweenness_comparison <- data.frame(Friendship=friendship_centrality[order(-friendship_centrality$Betweenness), ][1:5, c("Name", "Betweenness")],
                                Peer_Tutor=tutor_centrality[order(-tutor_centrality$Betweenness), ][1:5, c("Name", "Betweenness")])

betweenness_comparison